# Percolation Threshold Readability Model Demo

## Overview

This notebook demonstrates the Percolation Threshold Readability Model.

In [ ]:
import json, os

GITHUB_DATA_URL = 'https://raw.githubusercontent.com/AMGrobelnik/ai-invention-703cc9-network-percolation-features-for-text-re/main/round-1/experiment-1/demo/mini_demo_data.json'

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists('mini_demo_data.json'):
        with open('mini_demo_data.json') as f:
            return json.load(f)
    raise FileNotFoundError('Could not load mini_demo_data.json')

data = load_data()
print(f'Loaded {len(data["examples"])} examples')

In [ ]:
CONFIG = {'num_random_orderings': 5, 'sbert_threshold': 0.5}
print('Config loaded')


## Step 1: Data Preparation


In [ ]:
from dataclasses import dataclass
from typing import Dict, Any

@dataclass
class TextData:
    text_id: str
    content: str
    grade_level: float
    metadata: Dict[str, Any]

texts = []
for example in data['examples'][:12]:
    texts.append(TextData(
        text_id=example['text_id'],
        content=example['content'],
        grade_level=example['grade_level'],
        metadata=example.get('metadata', {})
    ))

print(f'Prepared {len(texts)} text examples')


## Step 2: Cohesion Network Construction

Build networks from text using TF-IDF similarity.


In [ ]:
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
import networkx as nx

class CohesionNetworkBuilder:
    def __init__(self):
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()
    
    def segment_sentences(self, text):
        sentences = sent_tokenize(text)
        return [s.strip() for s in sentences if len(s.split()) >= 5]
    
    def build_network(self, text):
        sentences = self.segment_sentences(text)
        if len(sentences) < 2:
            G = nx.Graph()
            G.add_nodes_from(range(len(sentences)))
            return G, sentences
        # Compute edges using TF-IDF
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(sentences)
        similarity = (tfidf_matrix * tfidf_matrix.T).toarray()
        G = nx.Graph()
        G.add_nodes_from(range(len(sentences)))
        for i in range(len(sentences)):
            for j in range(i+1, len(sentences)):
                if similarity[i, j] > 0.5:
                    G.add_edge(i, j, weight=similarity[i, j])
        return G, sentences

network_builder = CohesionNetworkBuilder()
print('Network builder initialized')


## Step 3: Percolation Threshold Computation


In [ ]:
import random

class PercolationAnalyzer:
    def __init__(self, num_random_orderings=5):
        self.num_random_orderings = num_random_orderings
    
    def compute_percolation_threshold(self, G):
        if G.number_of_nodes() < 2:
            return 0.0, 0.0
        edges = list(G.edges())
        n_nodes = G.number_of_nodes()
        threshold_target = n_nodes // 2
        p_c_values = []
        for _ in range(self.num_random_orderings):
            random.shuffle(edges)
            parent = list(range(n_nodes))
            component_size = [1] * n_nodes
            max_size = 1
            p_c = 1.0
            for k, (u, v) in enumerate(edges):
                root_u = self._find(parent, u)
                root_v = self._find(parent, v)
                if root_u != root_v:
                    if component_size[root_u] < component_size[root_v]:
                        root_u, root_v = root_v, root_u
                    parent[root_v] = root_u
                    component_size[root_u] += component_size[root_v]
                    if component_size[root_u] > max_size:
                        max_size = component_size[root_u]
                if max_size >= threshold_target:
                    p_c = (k + 1) / len(edges)
                    break
            p_c_values.append(p_c)
        return float(np.mean(p_c_values)), float(np.std(p_c_values))
    
    def _find(self, parent, x):
        if parent[x] != x:
            parent[x] = self._find(parent, parent[x])
        return parent[x]

percolation_analyzer = PercolationAnalyzer(num_random_orderings=CONFIG['num_random_orderings'])
print('Percolation analyzer initialized')


## Step 4: Process All Texts

For each text, build network, compute p_c, and compute traditional readability metrics.


In [ ]:
import textstat

class BaselineReadabilityMetrics:
    def compute_all_metrics(self, text):
        metrics = {}
        try:
            metrics['flesch_kincaid'] = textstat.flesch_kincaid_grade(text)
        except: metrics['flesch_kincaid'] = 0.0
        try:
            metrics['dale_chall'] = textstat.dale_chall_readability_score(text)
        except: metrics['dale_chall'] = 0.0
        return metrics

baseline_metrics = BaselineReadabilityMetrics()
results = []

for i, text_data in enumerate(texts):
    print(f"Processing text {i+1}/{len(texts)}: {text_data.text_id}")
    result = {'text_id': text_data.text_id, 'grade_level': text_data.grade_level, 'n_sentences': 0, 'n_edges': 0}
    G, sentences = network_builder.build_network(text_data.content)
    result['n_sentences'] = len(sentences)
    result['n_edges'] = G.number_of_edges()
    if len(sentences) >= 2:
        p_c_mean, p_c_std = percolation_analyzer.compute_percolation_threshold(G)
        result['p_c_mean'] = p_c_mean
        result['p_c_std'] = p_c_std
    else:
        result['p_c_mean'] = 0.0
        result['p_c_std'] = 0.0
    baseline = baseline_metrics.compute_all_metrics(text_data.content)
    result.update(baseline)
    results.append(result)

df = pd.DataFrame(results)
print(f"\nProcessed {len(df)} texts")
print(df[['text_id', 'grade_level', 'n_sentences', 'n_edges', 'p_c_mean']].head())



## Step 5: Correlation Analysis

Analyze correlation between p_c and grade level.


In [ ]:
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Compute correlation
valid_data = df.dropna(subset=['p_c_mean', 'grade_level'])
if len(valid_data) > 2:
    pearson_r, p_value = stats.pearsonr(valid_data['p_c_mean'], valid_data['grade_level'])
    print(f"Correlation (p_c vs grade): r = {pearson_r:.3f}, p = {p_value:.3f}")

# Simple regression with p_c only
if len(valid_data) >= 10:
    X = valid_data[['p_c_mean']].values
    y = valid_data['grade_level'].values
    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    print(f"Simple Model (p_c only): R2 = {r2:.3f}, RMSE = {rmse:.3f}")



## Step 6: Visualization of Results

Visualize the relationship between percolation threshold and grade level.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Percolation Threshold Readability Model - Demo Results', fontsize=14, fontweight='bold')

# Plot 1: p_c vs Grade Level
ax1 = axes[0, 0]
valid_data = df.dropna(subset=['p_c_mean', 'grade_level'])
if len(valid_data) > 0:
    ax1.scatter(valid_data['grade_level'], valid_data['p_c_mean'], s=100, alpha=0.7, edgecolors='black', linewidth=1.5)
    ax1.set_xlabel('Grade Level')
    ax1.set_ylabel('Percolation Threshold (p_c)')
    ax1.set_title('Percolation Threshold vs Grade Level')
    ax1.grid(True, alpha=0.3)

# Plot 2: Network statistics
ax2 = axes[0, 1]
if 'n_sentences' in df.columns and 'n_edges' in df.columns:
    ax2.scatter(df['n_sentences'], df['n_edges'], s=100, alpha=0.7, edgecolors='black', linewidth=1.5)
    ax2.set_xlabel('Number of Sentences')
    ax2.set_ylabel('Number of Edges')
    ax2.set_title('Network Size Statistics')
    ax2.grid(True, alpha=0.3)

# Plot 3: Summary text
ax3 = axes[1, 0]
ax3.axis('off')
summary_text = "=== MODEL PERFORMANCE ===\n\n"
if len(valid_data) > 2:
    pearson_r, p_value = stats.pearsonr(valid_data['p_c_mean'], valid_data['grade_level'])
    summary_text += f"Correlation (p_c vs grade): r = {pearson_r:.3f}, p = {p_value:.3f}\n\n"
if len(valid_data) >= 10:
    X = valid_data[['p_c_mean']].values
    y = valid_data['grade_level'].values
    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    summary_text += f"Simple Model (p_c only): R2 = {r2:.3f}, RMSE = {rmse:.3f}\n"
ax3.text(0.1, 0.9, summary_text, fontsize=11, family='monospace', verticalalignment='top', transform=ax3.transAxes, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 4: Detailed results table
ax4 = axes[1, 1]
ax4.axis('off')
table_text = df[['text_id', 'grade_level', 'p_c_mean', 'flesch_kincaid']].to_string(index=False)
ax4.text(0.1, 0.9, table_text, fontsize=9, family='monospace', verticalalignment='top', transform=ax4.transAxes)

plt.tight_layout()
plt.show()



## Summary

This demo has shown the Percolation Threshold Readability Model.

### Key Insights:
- **High cohesion texts** (simple, repetitive) → **low p_c**
- **Low cohesion texts** (complex, varied) → **high p_c**

### Next Steps:
- Scale up: Increase `num_random_orderings` to 50 for stable p_c estimates
- Add more examples across grade levels
